# Multithreading Practical Implementation With Python

This notebook shows how to build practical multithreading programs in Python. It focuses on patterns you can reuse for I/O-heavy tasks such as downloading files, waiting on services, or processing many small jobs in parallel.

## Learning Goals

- Create and start Python threads
- Wait for threads to finish with `join()`
- Share results safely between threads
- Use `Lock` to protect shared state
- Build a simple worker pool with `queue.Queue`

In [1]:
import threading
import queue
import time
from time import perf_counter

## Example 1: Serial Work vs Threaded Work

The first example compares running several slow tasks one after another with running them as threads. The tasks use `sleep()` to simulate waiting on I/O.

In [2]:
def slow_job(name, delay):
    print(f'{name} started')
    time.sleep(delay)
    print(f'{name} finished')
    return f'{name} result'

jobs = [(f'Job-{index}', 1) for index in range(1, 4)]

start = perf_counter()
for name, delay in jobs:
    slow_job(name, delay)
serial_time = perf_counter() - start

print(f'Serial execution took {serial_time:.2f} seconds')

Job-1 started
Job-1 finished
Job-2 started
Job-2 finished
Job-3 started
Job-3 finished
Serial execution took 3.00 seconds


In [3]:
results = []
results_lock = threading.Lock()

def threaded_job(name, delay):
    print(f'{name} started on {threading.current_thread().name}')
    time.sleep(delay)
    result = f'{name} result'
    with results_lock:
        results.append(result)
    print(f'{name} finished on {threading.current_thread().name}')

threads = [
    threading.Thread(target=threaded_job, args=(name, delay), name=f'Worker-{index}')
    for index, (name, delay) in enumerate(jobs, start=1)
]

start = perf_counter()
for thread in threads:
    thread.start()

for thread in threads:
    thread.join()
threaded_time = perf_counter() - start

print(f'Threaded execution took {threaded_time:.2f} seconds')
print('Collected results:', results)

Job-1 started on Worker-1
Job-2 started on Worker-2
Job-3 started on Worker-3
Job-1 finished on Worker-1
Job-2 finished on Worker-2
Job-3 finished on Worker-3
Threaded execution took 1.00 seconds
Collected results: ['Job-1 result', 'Job-2 result', 'Job-3 result']


## Example 2: Protect Shared Data With a Lock

When multiple threads update the same variable, a lock keeps the update safe. Without a lock, the final value can be inconsistent because two threads may read and write at the same time.

In [4]:
counter = 0
counter_lock = threading.Lock()

def increment_counter(repeat_count):
    global counter
    for _ in range(repeat_count):
        with counter_lock:
            counter += 1

repeat_count = 100000
worker_threads = [
    threading.Thread(target=increment_counter, args=(repeat_count,))
    for _ in range(4)
]

for thread in worker_threads:
    thread.start()

for thread in worker_threads:
    thread.join()

print('Expected counter value:', repeat_count * 4)
print('Actual counter value:', counter)

Expected counter value: 400000
Actual counter value: 400000


## Example 3: Worker Pool With `queue.Queue`

A queue is a clean way to distribute work across threads. Each worker takes one item, processes it, and marks it done. This pattern is useful for batch jobs and background tasks.

In [5]:
task_queue = queue.Queue()
processed_tasks = []
processed_lock = threading.Lock()

def queue_worker():
    while True:
        item = task_queue.get()
        if item is None:
            task_queue.task_done()
            break
        name, delay = item
        print(f'{threading.current_thread().name} processing {name}')
        time.sleep(delay)
        with processed_lock:
            processed_tasks.append(name)
        task_queue.task_done()

workers = [threading.Thread(target=queue_worker, name=f'QueueWorker-{index}') for index in range(1, 4)]

for worker in workers:
    worker.start()

for job in jobs:
    task_queue.put(job)

for _ in workers:
    task_queue.put(None)

task_queue.join()

for worker in workers:
    worker.join()

print('Processed tasks:', processed_tasks)

QueueWorker-1 processing Job-1QueueWorker-2 processing Job-2
QueueWorker-3 processing Job-3

Processed tasks: ['Job-2', 'Job-3', 'Job-1']


## Practical Takeaways

- Use threads for waiting-heavy work, not for CPU-heavy number crunching.
- Always use `join()` when the main program must wait for threads.
- Protect shared state with a lock, queue, or another synchronization primitive.
- Keep thread workers small and focused so they are easy to test and debug.